# Assigment

In diesem Assignment besprechen wir die folgenden  Python-Grundlagen

- Pandas
- Einbinden von Daten mit APIs
- Einbinden von LLMs
- Bauen von Dashboards



## Pandas

Pandas ist eine Python-Bibliothek zur Datenmanipulation und -analyse, die tabellarische Datenstrukturen wie DataFrames bereitstellt und effizientes Filtern, Transformieren und Auswerten von Daten ermöglicht.

Im Folgenden erarbeiten wir uns anhand von Beispielen die wichtigsten Syntax-Grundlagen:

In [ ]:
# 1) Pandas: minimales Beispiel mit Kern-Syntax
import pandas as pd

df = pd.DataFrame({
    "company": ["Apple", "Microsoft", "NVIDIA", "Amazon", "Meta"],
    "revenue_usd_bn": [383, 212, 61, 575, 135],
    "year": [2024, 2024, 2024, 2024, 2024]
})

print("DataFrame:")
print(df)

# Auswahl der Spalten:
print("\nNur zwei Spalten:")
print(df[["company", "revenue_usd_bn"]])

# Eine einzelne Spalte ohne DF-Struktur, nennen wir Panda-Series:
print(type(df["company"]))
print(df["company"])

print(type(df[["company"]]))
print(df[["company"]])

# Auswahl an Zeilen über eine boolische Bedingung:
print("\nFilter (Umsatz > 150):")
print(df[df["revenue_usd_bn"] > 150])

print("\nEinfache Kennzahl (Mittelwert Umsatz):", round(df["revenue_usd_bn"].mean(), 2))

# Neue Spalte: Anteil am Gesamtmarkt in Prozent
total_market = df["revenue_usd_bn"].sum()
df["market_share_pct"] = (df["revenue_usd_bn"] / total_market * 100).round(2)

print("\nMit neuer Spalte market_share_pct:")
print(df[["company", "revenue_usd_bn", "market_share_pct"]])

# loc: label-basiert (Zeilenbedingung + Spaltennamen)
print("\nloc-Beispiel (Umsatz > 150, nur company und market_share_pct):")
print(df.loc[df["revenue_usd_bn"] > 150, ["company", "market_share_pct"]])

# iloc: positionsbasiert (erste 3 Zeilen, erste 3 Spalten)
print("\niloc-Beispiel (erste 3 Zeilen/3 Spalten):")
print(df.iloc[:3, :3])


total_market = df["revenue_usd_bn"].sum()
df["market_share_pct"] = (df["revenue_usd_bn"] / total_market * 100).round(2)

DataFrame:
     company  revenue_usd_bn  year
0      Apple             383  2024
1  Microsoft             212  2024
2     NVIDIA              61  2024
3     Amazon             575  2024
4       Meta             135  2024

Nur zwei Spalten:
     company  revenue_usd_bn
0      Apple             383
1  Microsoft             212
2     NVIDIA              61
3     Amazon             575
4       Meta             135

Filter (Umsatz > 150):
     company  revenue_usd_bn  year
0      Apple             383  2024
1  Microsoft             212  2024
3     Amazon             575  2024

Einfache Kennzahl (Mittelwert Umsatz): 273.2

Mit neuer Spalte market_share_pct:
     company  revenue_usd_bn  market_share_pct
0      Apple             383             28.04
1  Microsoft             212             15.52
2     NVIDIA              61              4.47
3     Amazon             575             42.09
4       Meta             135              9.88

loc-Beispiel (Umsatz > 150, nur company und market_share_

### Übungen zu Pandas

Bearbeite die folgenden Aufgaben direkt untereinander in einer neuen Code-Zelle:

1. Gib nur die Spalten `company` und `year` aus.
2. Filtere alle Unternehmen mit `revenue_usd_bn > 200`.
3. Berechne den Median von `revenue_usd_bn`.
4. Lege eine neue Spalte `revenue_eur_bn` an (vereinfachte Annahme: `USD * 0.92`).
5. Nutze `loc`, um nur `company` und `market_share_pct` fuer Unternehmen mit `market_share_pct > 20` anzuzeigen.
6. Nutze `iloc`, um die ersten 2 Zeilen und die ersten 3 Spalten auszugeben.
7. Sortiere den DataFrame nach `market_share_pct` absteigend und zeige nur `company` + `market_share_pct`.
8. Erstellt eine weitere Spalte `is_large_player` mit `True/False` fuer `market_share_pct > 15`.

In [12]:
df[["company", "year"]]

df.loc[df["revenue_usd_bn"] > 150, ["company", "market_share_pct"]]
df[["revenue_usd_bn"]].median()
df["market_share_vgl_ms"] = df["revenue_usd_bn"] / df.loc[df["company"] == "Microsoft", "revenue_usd_bn"].values[0]

## Einbindung der Edgar API

In [22]:
# 2) Edgar API: Minimaler Abruf von Finanzdaten (SEC companyfacts)
import requests
from main_api import get_core_metrics_from_companyfacts
from getpass import getpass

# CIK ist ein Aktien-Identifier (ähnlich einer WKN)
def call_edgar_api(cik):
    cik10 = str(cik).zfill(10)
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik10}.json"
    headers = {"User-Agent": "test@beispiel.de"}
    response = requests.get(url, headers=headers, timeout=20)
    response.raise_for_status()
    companyfacts = response.json()
    return companyfacts

companyfacts = call_edgar_api(cik = "0000320193")
print("Company:", companyfacts.get("entityName"))
print("Beispielhafte Top-Level Keys:", list(companyfacts.keys())[:6])

Company: Apple Inc.
Beispielhafte Top-Level Keys: ['cik', 'entityName', 'facts']


- Untersuchen Sie nun companyfacts
- Wo verstecken sich Bilanzdaten?

Als nächstes erstellen ich eine ganz einfach Dash-GUI, in der einige abgefragte Daten grafisch dargestellt werden.

In [14]:
BIG_TECH = {
    "Apple": 320193,
    "Microsoft": 789019,
    "Alphabet": 1652044
}

### Mini-Dash-Demo: Auswahl Big-Tech und Visualisierung von Jahresueberschuss, FCF und CAPEX

Hinweis: In Jupyter wird der Link auf den lokalen Dash-Server angezeigt. Zum Starten die Zelle ausfuehren und dann den Link oeffnen.

In [19]:
# Sehr kompakt: companystats mit 5 Bilanzkennzahlen
import pandas as pd

selected_company = "Apple"
cf = call_edgar_api(cik=BIG_TECH[selected_company])

last = lambda tag: next(
    (
        x for x in sorted(
            cf.get("facts", {}).get("us-gaap", {}).get(tag, {}).get("units", {}).get("USD", []),
            key=lambda r: ((r.get("fy") or 0), (r.get("filed") or "")),
            reverse=True,
        )
        if x.get("form") == "10-K" and x.get("val") is not None
    ),
    {},
)

companystats = pd.DataFrame([
    {"metric": "Assets",         "us_gaap_tag": "Assets",                                "fiscal_year": (r := last("Assets")).get("fy"),                                "value_usd_bn": round(r.get("val") / 1e9, 2) if r.get("val") is not None else None},
    {"metric": "Liabilities",    "us_gaap_tag": "Liabilities",                           "fiscal_year": (r := last("Liabilities")).get("fy"),                           "value_usd_bn": round(r.get("val") / 1e9, 2) if r.get("val") is not None else None},
    {"metric": "Equity",         "us_gaap_tag": "StockholdersEquity",                    "fiscal_year": (r := last("StockholdersEquity")).get("fy"),                    "value_usd_bn": round(r.get("val") / 1e9, 2) if r.get("val") is not None else None},
    {"metric": "Cash",           "us_gaap_tag": "CashAndCashEquivalentsAtCarryingValue", "fiscal_year": (r := last("CashAndCashEquivalentsAtCarryingValue")).get("fy"), "value_usd_bn": round(r.get("val") / 1e9, 2) if r.get("val") is not None else None},
    {"metric": "Short-term debt","us_gaap_tag": "LongTermDebtCurrent",                   "fiscal_year": (r := last("LongTermDebtCurrent")).get("fy"),                   "value_usd_bn": round(r.get("val") / 1e9, 2) if r.get("val") is not None else None},
])

display(companystats)

,metric,us_gaap_tag,fiscal_year,value_usd_bn
0,Assets,Assets,2025,364.98
1,Liabilities,Liabilities,2025,308.03
2,Equity,StockholdersEquity,2025,50.67
3,Cash,CashAndCashEquivalentsAtCarryingValue,2025,29.94
4,Short-term debt,LongTermDebtCurrent,2025,10.91


In [20]:
# MINI Dash: Dropdown-CIK + dynamische Company-Stats-Tabelle
from dash import Dash, dcc, html, Input, Output
import pandas as pd

def latest_10k(companyfacts, tag):
    vals = companyfacts.get("facts", {}).get("us-gaap", {}).get(tag, {}).get("units", {}).get("USD", [])
    vals = [v for v in vals if v.get("form") == "10-K" and v.get("val") is not None]
    vals = sorted(vals, key=lambda v: ((v.get("fy") or 0), (v.get("filed") or "")), reverse=True)
    return vals[0] if vals else {}

def company_stats_from_cik(cik10):
    cf = call_edgar_api(cik=cik10)
    metrics = [
        ("Assets", "Assets"),
        ("Liabilities", "Liabilities"),
        ("Equity", "StockholdersEquity"),
        ("Cash", "CashAndCashEquivalentsAtCarryingValue"),
        ("Short-term debt", "LongTermDebtCurrent"),
    ]
    rows = []
    for label, tag in metrics:
        r = latest_10k(cf, tag)
        rows.append({
            "metric": label,
            "fiscal_year": r.get("fy"),
            "value_usd_bn": round(r.get("val") / 1e9, 2) if r.get("val") is not None else None,
        })
    return pd.DataFrame(rows)

COMPANY_OPTIONS = [{"label": f"{name} ({str(cik).zfill(10)})", "value": str(cik).zfill(10)} for name, cik in BIG_TECH.items()]
CIK_TO_NAME = {str(cik).zfill(10): name for name, cik in BIG_TECH.items()}

app = Dash(__name__)
app.layout = html.Div([
    html.H4("Mini Dash: Company Stats"),
    dcc.Dropdown(id="cik_dropdown", options=COMPANY_OPTIONS, value=COMPANY_OPTIONS[0]["value"], clearable=False),
    html.Div(id="selected_info", style={"marginTop": "10px", "fontWeight": "bold"}),
    html.Div(id="stats_table", style={"marginTop": "8px"})
])

@app.callback(
    Output("selected_info", "children"),
    Output("stats_table", "children"),
    Input("cik_dropdown", "value")
)
def render_table(selected_cik10):
    try:
        df_stats = company_stats_from_cik(selected_cik10)
        name = CIK_TO_NAME.get(selected_cik10, "Unknown")
    except Exception as e:
        return "", html.Div(f"Fehler beim Laden der Daten: {e}", style={"color": "crimson"})

    header = html.Tr([html.Th(c, style={"border": "1px solid #ccc", "padding": "6px"}) for c in df_stats.columns])
    body = [
        html.Tr([html.Td(df_stats.iloc[i][c], style={"border": "1px solid #ccc", "padding": "6px"}) for c in df_stats.columns])
        for i in range(len(df_stats))
    ]
    table = html.Table([html.Thead(header), html.Tbody(body)], style={"borderCollapse": "collapse", "width": "100%"})
    return f"Ausgewaehlt: {name} ({selected_cik10})", table

print("Open Dash app: http://127.0.0.1:8052")
try:
    app.run(debug=False, port=8052, jupyter_mode="external")
except TypeError:
    app.run(debug=False, port=8052)

Open Dash app: http://127.0.0.1:8052
Dash app running on http://127.0.0.1:8052/


# Einbindung eines LLMs

Sie müssen sich zuerst einen Api-Key auf Openrouter machen. 

In [23]:
import os
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
if not openrouter_api_key:
    print("Bitte setzen Sie die Variable OPENROUTER_API_KEY mit Ihrem OpenRouter API-Schlüssel.")
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API Key eingeben (wird nicht angezeigt): ")

def _build_openrouter_url() -> str:
    """Akzeptiert verschiedene Base-URL-Formate und erzeugt immer den korrekten Endpoint."""
    raw = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1/chat/completions").strip()
    base = raw.rstrip("/")

    if base.endswith("/chat/completions"):
        return base
    if base.endswith("/api/v1"):
        return f"{base}/chat/completions"
    if base == "https://openrouter.ai":
        return "https://openrouter.ai/api/v1/chat/completions"
    if "openrouter.ai" in base and "/api/v1" not in base:
        return f"{base}/api/v1/chat/completions"
    return base



Bitte setzen Sie die Variable OPENROUTER_API_KEY mit Ihrem OpenRouter API-Schlüssel.


Jetzt können wir das LLM ansprechen. Die nächste 

In [27]:
def ask_llm(question: str, context: str = "") -> str:
    """Cloud-Free-Tier ueber OpenRouter."""
    api_key = os.getenv("OPENROUTER_API_KEY", "")
    if not api_key:
        return "Kein OPENROUTER_API_KEY gesetzt."

    url = _build_openrouter_url()
    model = os.getenv("OPENROUTER_MODEL", "openrouter/auto")

    prompt = f"Kontext:\n{context}\n\nFrage: {question}\nAntworte kurz und klar."

    # --- das fehlte komplett ---
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    payload = {
        "model": model,
        "messages": [
            {"role": "user", "content": prompt}
        ]
    }

    response = requests.post(url, headers=headers, json=payload, timeout=30)
    response.raise_for_status()
    return response.json()["choices"][0]["message"]["content"]


In [28]:
question = input("Stellen Sie eine Frage an einen Finanzberater?")
kontext = "Sie sind ein Finanzberater, spezialisiert auf Privatkundenberatung und antworten in einem höflichen, aber lockeren Ton."
answer = ask_llm(question = question, context=kontext)
print("Antwort des LLM:", answer)

Antwort des LLM: Ja – aber nicht pauschal. Aktien können langfristig sinnvoll sein, vor allem als Teil einer diversifizierten Strategie mit Risikotoleranz und Geld, das Sie nicht kurzfristig brauchen. Viele Privatkunden setzen lieber auf breit gestreute ETFs statt Einzelaktien.
